In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

# Load and preprocess MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train = x_train[..., tf.newaxis]  # (N, 28, 28, 1)
x_test = x_test[..., tf.newaxis]

# Build CNN
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(10, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_accuracy")
    ]
)

# Train
history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

# Evaluate on test set (returns all compiled metrics)
eval_results = model.evaluate(x_test, y_test, verbose=0, return_dict=True)
print("\nTest evaluation:")
for metric_name, metric_value in eval_results.items():
    print(f"{metric_name}: {metric_value:.4f}")

# Predictions for additional classification metrics
y_prob = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

# Confusion matrix and macro precision/recall/F1
cm = tf.math.confusion_matrix(y_test, y_pred, num_classes=10).numpy()
tp = np.diag(cm)
fp = cm.sum(axis=0) - tp
fn = cm.sum(axis=1) - tp

precision_per_class = np.divide(tp, tp + fp, out=np.zeros_like(tp, dtype=float), where=(tp + fp) != 0)
recall_per_class = np.divide(tp, tp + fn, out=np.zeros_like(tp, dtype=float), where=(tp + fn) != 0)
f1_per_class = np.divide(
    2 * precision_per_class * recall_per_class,
    precision_per_class + recall_per_class,
    out=np.zeros_like(precision_per_class, dtype=float),
    where=(precision_per_class + recall_per_class) != 0
)

print("\nMacro metrics:")
print(f"precision: {precision_per_class.mean():.4f}")
print(f"recall:    {recall_per_class.mean():.4f}")
print(f"f1-score:  {f1_per_class.mean():.4f}")

print("\nPer-class metrics:")
for cls in range(10):
    print(
        f"Class {cls}: "
        f"precision={precision_per_class[cls]:.4f}, "
        f"recall={recall_per_class[cls]:.4f}, "
        f"f1={f1_per_class[cls]:.4f}"
    )

print("\nConfusion matrix:")
print(cm)

Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 17s 18ms/step - accuracy: 0.8548 - loss: 0.4571 - top3_accuracy: 0.9470 - val_accuracy: 0.9838 - val_loss: 0.0555 - val_top3_accuracy: 0.9985
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - accuracy: 0.9767 - loss: 0.0753 - top3_accuracy: 0.9980 - val_accuracy: 0.9898 - val_loss: 0.0369 - val_top3_accuracy: 0.9988
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - accuracy: 0.9828 - loss: 0.0534 - top3_accuracy: 0.9985 - val_accuracy: 0.9917 - val_loss: 0.0346 - val_top3_accuracy: 0.9992
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - accuracy: 0.9880 - loss: 0.0398 - top3_accuracy: 0.9993 - val_accuracy: 0.9893 - val_loss: 0.0384 - val_top3_accuracy: 0.9985
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - accuracy: 0.9898 - loss: 0.0327 - top3_accuracy: 0.9995 - val_accuracy: 0.9903 - val_loss: 0.0317 - val_top3_accuracy: 0.9995

Test evaluation:
accuracy: 0.9898
loss: 0.0288
top3_accuracy: 0.9997

Macro metrics:
precision